# Train & benchmark everything (Google Colab)
**First:** `Runtime → Change runtime type → T4 GPU`. Then run the cells in order (about 40–60 minutes in total).

This notebook produces **every number** in the homework: four models, distillation, quantization and the results table.
Latency is measured on the **CPU**, not on the GPU, because the benchmark runs all models on the CPU.

In [ ]:
# 1) Clone the repository and install the packages
!git clone https://github.com/berkotun/support-ticket-router.git
%cd support-ticket-router
!pip install -q -r requirements.txt

In [ ]:
# 2) Download CLINC150, save the splits, build the typo test set and the spelling vocabulary
!python -m src.prepare_data

In [ ]:
# 3) Area B: tf-idf + logistic regression baseline
!python -m src.train_tfidf

In [ ]:
# 4) Area A: own fastText embeddings + feedforward network (SGD + backpropagation)
!python -m src.train_fasttext_ffn

In [ ]:
# 5) Area C: fine-tune BERT-base (teacher) on the GPU
!python -m src.train_teacher

In [ ]:
# 6) Area D: knowledge distillation into DistilBERT (student) on the GPU
!python -m src.distill

In [ ]:
# 7) Area D: dynamic INT8 quantization (student and teacher)
!python -m src.quantize

In [ ]:
# 8) Evaluate all models on the SAME test set (CPU latency included, ~15-25 min)
!python -m src.benchmark

In [ ]:
# 9) Look at the results
from IPython.display import Markdown, display
display(Markdown(open('results/results_table.md').read()))
import pandas as pd
pd.read_csv('results/failures.csv').head(15)

In [ ]:
# 10) Log in to the Hugging Face Hub (the app downloads the models from there)
#     Create a token with 'write' access at https://huggingface.co/settings/tokens
#     and paste it into the box that appears below this cell.
from getpass import getpass
from huggingface_hub import login
login(token=getpass("Hugging Face token: "))

In [ ]:
!python -m src.upload_models berktn/support-ticket-router-models

In [ ]:
# 11) Download the results folder and add it to the GitHub repository (evidence for every number)
!cp models/thresholds.json results/
!zip -r results.zip results
from google.colab import files
files.download('results.zip')

In [ ]:
# (Optional) Try the app inside Colab through a temporary public link
!sed -i 's/demo.launch()/demo.launch(share=True)/' app.py
!python app.py